In [0]:
catalog_name = "crypto"
schema_name = "bronze"
schema_silver = "silver"
volume_name = "binance_raw_data"

"""
Creation <catalogue>.<schema>.<volume> if not exists
"""

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_silver}")

display(spark.sql(f"DESCRIBE VOLUME {catalog_name}.{schema_name}.{volume_name}"))

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

display(spark.sql(f"SHOW TABLES"))

In [0]:
spark.catalog.setCurrentCatalog(catalog_name)
spark.catalog.setCurrentDatabase(schema_name)

# Display available tables in your schema
spark.catalog.listTables(schema_name)

In [0]:
%sql
SHOW VOLUMES IN crypto.bronze;

In [0]:
%sql
USE CATALOG crypto;
USE SCHEMA bronze;

In [0]:
spark.sql(f"LIST '/Volumes/{catalog_name}/{schema_name}/{volume_name}/' ").display()

In [0]:
path_daily = "/Volumes/crypto/bronze/binance_raw_data/batch/BTCUSDT/1d/*.csv"


spark.sql(f"DROP TABLE IF EXISTS crypto.silver.crypto_1d")

spark.sql(f"""
CREATE OR REPLACE TABLE crypto.silver.crypto_1d
AS
SELECT 
    -- Conversion du timestamp Binance (ms) en Timestamp Spark
    FROM_UNIXTIME(open_time / 1000) AS open_time,
    CAST(open AS DOUBLE) AS open_price,
    CAST(high AS DOUBLE) AS high_price,
    CAST(low AS DOUBLE) AS low_price,
    CAST(close AS DOUBLE) AS close_price,
    CAST(volume AS DOUBLE) AS volume,
    FROM_UNIXTIME(close_time / 1000) AS close_time,
    CAST(quote_volume AS DOUBLE) AS quote_volume,
    CAST(count AS INT) AS trade_count,
    CAST(taker_buy_volume AS DOUBLE) AS taker_buy_volume,
    CAST(taker_buy_quote_volume AS DOUBLE) AS taker_buy_quote_volume,
    split(_metadata.file_path, '/')[6] AS symbol
FROM read_files(
    "/Volumes/crypto/bronze/binance_raw_data/batch/*/1d/*.csv",
    format => 'csv',
    header => true,
    inferSchema => true
);
""")

display(spark.table("crypto.silver.crypto_1d").head(10))




In [0]:
spark.sql(f"DROP TABLE IF EXISTS crypto.silver.crypto_1h")

spark.sql(f"""
CREATE OR REPLACE TABLE crypto.silver.crypto_1h
AS
SELECT 
    -- Convert Timestamp Binance (ms) to Timestamp Spark
    FROM_UNIXTIME(open_time / 1000) AS open_time,
    CAST(open AS DOUBLE) AS open_price,
    CAST(high AS DOUBLE) AS high_price,
    CAST(low AS DOUBLE) AS low_price,
    CAST(close AS DOUBLE) AS close_price,
    CAST(volume AS DOUBLE) AS volume,
    FROM_UNIXTIME(close_time / 1000) AS close_time,
    CAST(quote_volume AS DOUBLE) AS quote_volume,
    CAST(count AS INT) AS trade_count,
    CAST(taker_buy_volume AS DOUBLE) AS taker_buy_volume,
    CAST(taker_buy_quote_volume AS DOUBLE) AS taker_buy_quote_volume,
    split(_metadata.file_path, '/')[6] AS symbol
FROM read_files(
    "/Volumes/crypto/bronze/binance_raw_data/batch/*/1h/*.csv",
    format => 'csv',
    header => true,
    inferSchema => true
);
""")

display(spark.table("crypto.silver.crypto_1h").head(10))

In [0]:
%sql
USE SCHEMA silver;



In [0]:
spark.sql(f"SHOW TABLES;").display()